Consider the results from the first 100 of your experiments from notebook 04, that is, the number of resistant mutants m<sub>1</sub>, m<sub>2</sub>, ..., m<sub>100</sub>. Treat these 100 numbers as data. Now write a program that leverages results of new simulations to infer the most likely value of μ, the mutation rate, and, if possible, a 95% confidence interval for it. In doing so, pretend that you do not know what μ is in advance. Consider using two pieces of evidence:

1.   the number of cultures with no resistant mutants,
2.   the mean number of resistant mutants.

Is your data also consistent with the acquired resistance hypothesis? Why or why not?

In [ ]:
# Observed data
# Variables need to be imported from prior notebook

# Take first 100 cultures as "observed data"
data = mut_pois[:100]   # or mut_binom[:100]

# Summary statistics
n = len(data)
p0_obs = np.sum(data == 0) / n
mean_obs = np.mean(data)

print("Observed P0:", p0_obs)
print("Observed mean:", mean_obs)

In [ ]:
# Quick estimate with P(0) method

N = 2**15

m_est = -np.log(p0_obs)
mu_p0 = m_est / N

print("P0-based estimate of mu:", mu_p0)

In [ ]:
# Mean-based estimate method

mu_mean = mean_obs / N
print("Mean-based estimate of mu:", mu_mean)

In [ ]:
# Simulation-based inference

def simulate_ld(mu, C=100):
    mutants = np.zeros(C, dtype=int)
    sensitive = np.ones(C, dtype=int)

    g = 15

    for gen in range(g):
        divisions = sensitive

        # Poisson approximation
        new_mut = np.random.poisson(divisions * mu)

        sensitive = 2 * sensitive - new_mut
        mutants = 2 * mutants + new_mut

    return mutants

In [ ]:
# Play around with μ (grid search to estimate the best μ)

mu_grid = np.logspace(-9, -5, 40)

scores = []

for mu in mu_grid:
    sims = []

    # replicate simulations to reduce noise
    for _ in range(20):
        sim_data = simulate_ld(mu, C=100)
        p0_sim = np.sum(sim_data == 0) / 100
        mean_sim = np.mean(sim_data)

        # distance metric (combine both statistics)
        dist = (p0_sim - p0_obs)**2 + (mean_sim - mean_obs)**2
        sims.append(dist)

    scores.append(np.mean(sims))

# Best μ
best_mu = mu_grid[np.argmin(scores)]
print("Best-fit mu:", best_mu)

In [ ]:
# Find confidence intervals
# Approximate CI by selecting μ values that give near-optimal fits.

scores = np.array(scores)
min_score = np.min(scores)

# Accept μ values within 2x minimum error (tunable threshold)
accepted = mu_grid[scores < 2 * min_score]

ci_low = np.min(accepted)
ci_high = np.max(accepted)

print("95% CI (approx):", ci_low, "-", ci_high)

In [ ]:
# Visualize the fit

import matplotlib.pyplot as plt

plt.plot(mu_grid, scores)
plt.axvline(best_mu, color='r', linestyle='--', label='Best μ')
plt.xscale('log')
plt.xlabel("μ")
plt.ylabel("Fit error")
plt.title("Inference of mutation rate")
plt.legend()
plt.show()